# Qwen3-8B QLoRA SFT — Chess Logic GPT

Fine-tunes Qwen3-8B with 4-bit QLoRA on chess reasoning + formal logic data.

**Setup:**
1. Upload `train_mix.jsonl` and `eval_mix.jsonl` to a folder called `chess-logic-gpt` in your Google Drive.
2. Make the GitHub repo public OR upload `src/` to `Drive/chess-logic-gpt/src/`.
3. Run all cells top-to-bottom.
4. Checkpoints save to Drive automatically. Re-run to resume from the last checkpoint.

**Hardware:** T4 GPU (15GB) — fits 8B model in 4-bit (~6GB weights + optimizer).

In [ ]:
#@title 0. Setup Check
from google.colab import drive
import os

drive.mount('/content/drive')
d = '/content/drive/MyDrive/chess-logic-gpt'
if os.path.isdir(d):
    files = os.listdir(d)
    print('Drive/chess-logic-gpt/ contents:', files)
    need = {'train_mix.jsonl', 'eval_mix.jsonl'}
    have = need.intersection(files)
    missing = need - have
    if not missing:
        print('Data files found — ready to go!')
    else:
        print(f'MISSING: {missing}')
        print('Upload them to Drive/chess-logic-gpt/ then re-run this cell.')
else:
    print(f'Create folder "chess-logic-gpt" in Google Drive and add data files')

In [ ]:
#@title 1. Install dependencies
!pip install -q --upgrade-strategy only-if-needed \
    torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    transformers>=4.51 peft>=0.12 accelerate>=0.33 \
    datasets>=2.20 bitsandbytes python-chess orjson pyyaml zstandard

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU! Runtime > Change runtime type > T4 GPU')

In [ ]:
#@title 2. Copy data to local disk
import shutil

DRIVE_DATA = '/content/drive/MyDrive/chess-logic-gpt'
LOCAL_DATA = '/content/data/processed'
os.makedirs(LOCAL_DATA, exist_ok=True)

for name in ('train_mix.jsonl', 'eval_mix.jsonl', 'eval_puzzles_ood.jsonl'):
    src = f'{DRIVE_DATA}/{name}'
    dst = f'{LOCAL_DATA}/{name}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'Copied {name} ({os.path.getsize(dst) / 1e6:.1f} MB)')
    elif os.path.exists(dst):
        print(f'Already have {name}')
    else:
        print(f'Not found: {name}')

print('Local data:', os.listdir(LOCAL_DATA))

In [ ]:
#@title 3. Clone repo & install package
REPO = '/content/chess-logic-gpt'
if not os.path.isdir(f'{REPO}/src'):
    !git clone https://github.com/tdickers22-lgtm/chess-logic-gpt.git {REPO} 2>/dev/null
    if not os.path.isdir(f'{REPO}/src'):
        drive_src = f'{DRIVE_DATA}/src'
        if os.path.isdir(drive_src):
            os.makedirs(REPO, exist_ok=True)
            shutil.copytree(drive_src, f'{REPO}/src')
            print('Copied src/ from Google Drive')
        else:
            raise FileNotFoundError(
                'Cannot access repo. Make GitHub repo public or upload src/ to Drive/chess-logic-gpt/src/')

!pip install -q --no-deps -e {REPO}
print('Package installed from', REPO)

In [ ]:
#@title 4. Build training config
import yaml
from transformers.trainer_utils import get_last_checkpoint

OUTPUT_DIR = '/content/outputs/qwen3-8b-chess-logic-lora'
CKPT_DRIVE = f'{DRIVE_DATA}/checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DRIVE, exist_ok=True)

# Restore latest checkpoint from Drive
drive_ckpt = get_last_checkpoint(CKPT_DRIVE) if os.path.isdir(CKPT_DRIVE) else None
if drive_ckpt:
    ckpt_name = os.path.basename(drive_ckpt)
    local_ckpt = f'{OUTPUT_DIR}/{ckpt_name}'
    if not os.path.isdir(local_ckpt):
        shutil.copytree(drive_ckpt, local_ckpt)
        print(f'Restored from Drive: {ckpt_name}')
    else:
        print(f'Already local: {ckpt_name}')
else:
    print('No checkpoint on Drive — starting fresh')

config = {
    'model': {
        'base_model': 'Qwen/Qwen3-8B',
        'trust_remote_code': True,
        'load_in_4bit': True,
        'enable_thinking': False,
    },
    'data': {
        'train_file': f'{LOCAL_DATA}/train_mix.jsonl',
        'eval_file': f'{LOCAL_DATA}/eval_mix.jsonl',
        'text_field': 'text',
        'max_seq_length': 2048,
    },
    'lora': {
        'r': 32,
        'alpha': 64,
        'dropout': 0.05,
        'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                           'gate_proj', 'up_proj', 'down_proj'],
    },
    'training': {
        'output_dir': OUTPUT_DIR,
        'num_train_epochs': 1,
        'per_device_train_batch_size': 1,
        'per_device_eval_batch_size': 1,
        'gradient_accumulation_steps': 16,
        'learning_rate': 8e-5,
        'warmup_ratio': 0.03,
        'weight_decay': 0.01,
        'logging_steps': 10,
        'eval_steps': 200,
        'save_steps': 200,
        'save_total_limit': 2,
        'bf16': True,
        'gradient_checkpointing': True,
        'resume_from_checkpoint': True,
        'report_to': 'none',
    },
}

config_path = '/content/train_config.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'Config: {config_path}')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
#@title 5. Train
import time
START = time.time()

!cd {REPO} && python scripts/train_lora.py --config /content/train_config.yaml

elapsed = (time.time() - START) / 3600
print(f'Training took {elapsed:.2f} hours')

In [ ]:
#@title 6. Save checkpoint to Drive
import glob

ckpts = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint-*'))
if ckpts:
    latest = ckpts[-1]
    ckpt_name = os.path.basename(latest)
    drive_dst = f'{CKPT_DRIVE}/{ckpt_name}'
    if os.path.exists(drive_dst):
        shutil.rmtree(drive_dst)
    shutil.copytree(latest, drive_dst)
    print(f'Saved to Drive: {ckpt_name}')

    final_dst = f'{DRIVE_DATA}/final_model'
    if os.path.exists(final_dst):
        shutil.rmtree(final_dst)
    shutil.copytree(OUTPUT_DIR, final_dst, ignore=shutil.ignore_patterns('checkpoint-*'))
    print('Saved final model to Drive')
else:
    print('No checkpoints found')

print(f'\nDone! Re-run this notebook to continue training.')